# Data transformation

In [1]:
import pandas as pd
from pathlib import Path

## Membaca data sumber dan menyiapkan folder output

In [2]:
input_dir = Path.cwd().parent / 'data' / 'extracted'
output_dir = Path.cwd().parent / 'data' / 'transformed'
output_dir.mkdir(parents=True, exist_ok=True)

data = {file.stem: pd.read_parquet(file) for file in input_dir.glob('*.parquet')}
print(f'{len(data)} tabel berhasil dibaca.')

9 tabel berhasil dibaca.


## Transformasi nilai dan validasi kualitas data

## Customer

In [3]:
customers = data['stg_customers'].copy()
customers['customer_signup_date'] = pd.to_datetime(customers['customer_signup_date'], errors='coerce')
customers['last_order_date'] = pd.to_datetime(customers['last_order_date'], errors='coerce')
customers['age_imputed_flag'] = customers['age'].isna()
age_medians = customers.groupby('age_group', dropna=False)['age'].transform('median')
customers['age'] = customers['age'].fillna(age_medians).fillna(customers['age'].median())
customers['age_group'] = pd.cut(
    customers['age'], bins=[0, 24, 34, 44, 54, float('inf')],
    labels=['18-24', '25-34', '35-44', '45-54', '55+'], include_lowest=True
).astype('object').fillna('Unk/nown')
customers['preferred_device'] = customers['preferred_device'].fillna('Unknown')

## Customer Review

In [4]:
reviews = data['stg_customer_reviews'].copy()
reviews['review_date'] = pd.to_datetime(reviews['review_date'], errors='coerce')
reviews['sentiment_imputed_flag'] = reviews['review_sentiment'].isna()
sentiment_from_rating = pd.cut(
    reviews['rating'], bins=[0, 2, 3, 5], labels=['Negative', 'Neutral', 'Positive']
).astype('object')
reviews['review_sentiment'] = reviews['review_sentiment'].fillna(sentiment_from_rating)
reviews['review_date_batch_flag'] = reviews['review_date'].eq(pd.Timestamp('2025-12-31'))

## Marketing Campaign

In [5]:
campaigns = data['stg_marketing_campaigns'].copy()
for column in ['campaign_start_date', 'campaign_end_date']:
    campaigns[column] = pd.to_datetime(campaigns[column], errors='coerce')
campaigns['roi'] = ((campaigns['revenue_generated'] - campaigns['campaign_cost']) / campaigns['campaign_cost']).round(3)
campaigns['campaign_performance'] = campaigns['roi'].ge(0).map({True: 'Profitable', False: 'Loss Making'})

## Order

In [6]:
orders = data['stg_orders'].copy()
orders['order_date'] = pd.to_datetime(orders['order_date'], errors='coerce')
orders['coupon_code'] = orders['coupon_code'].fillna('No Coupon')
orders['campaign_id'] = orders['campaign_id'].fillna('No Campaign')
orders['final_amount_expected'] = (orders['subtotal'] + orders['shipping_fee'] + orders['tax_amount']).round(2)
orders['final_amount_check'] = orders['final_amount'].round(2).eq(orders['final_amount_expected'])
orders['discount_note'] = 'subtotal treated as net amount; discount not applied again'

## Order Item

In [7]:
order_items = data['stg_order_items'].copy()
order_items['profit_status'] = order_items['profit'].ge(0).map({True: 'Profitable', False: 'Loss Making'})

## Payment

In [8]:
payments = data['stg_payments'].copy()
payments['payment_date'] = pd.to_datetime(payments['payment_date'], errors='coerce')
payments['payment_validation'] = 'Valid'
payments.loc[(payments['payment_status'].isin(['Success', 'Refunded'])) & (payments['amount_paid'] <= 0), 'payment_validation'] = 'Check amount_paid'
payments.loc[(payments['payment_status'].isin(['Failed', 'Pending', 'Not Charged'])) & (payments['amount_paid'] != 0), 'payment_validation'] = 'Check zero amount'
payments.loc[(payments['payment_status'] == 'Refunded') & (payments['refund_amount'] <= 0), 'payment_validation'] = 'Check refund_amount'

## Product

In [9]:
products = data['stg_products'].copy()
products['product_launch_date'] = pd.to_datetime(products['product_launch_date'], errors='coerce')
products.loc[products['rating_count'].eq(0), 'rating_average'] = pd.NA
products['rating_validation'] = products['rating_average'].notna().map({True: 'Has rating', False: 'No rating'})
products['duplicate_name_flag'] = products['product_name'].duplicated(False)

## Return

In [10]:
returns = data['stg_returns'].copy()
returns['return_date'] = pd.to_datetime(returns['return_date'], errors='coerce')
returns['refund_validation'] = 'Valid'
returns.loc[returns['return_status'].isin(['Pending', 'Rejected']) & returns['refund_amount'].ne(0), 'refund_validation'] = 'Check refund amount'
returns.loc[(returns['return_status'] == 'Refund Completed') & returns['refund_amount'].le(0), 'refund_validation'] = 'Check refund amount'
order_status = orders[['order_id', 'order_status']].rename(columns={'order_status': 'related_order_status'})
returns = returns.merge(order_status, on='order_id', how='left')
returns['order_return_status_flag'] = returns['related_order_status'].ne('Returned')

## Shipments

In [11]:
shipments = data['stg_shipments'].copy()
for column in ['dispatch_date', 'expected_delivery_date', 'actual_delivery_date']:
    shipments[column] = pd.to_datetime(shipments[column], errors='coerce')
shipments['delivery_days_original'] = shipments['delivery_days']
shipments['delivery_days'] = (shipments['actual_delivery_date'] - shipments['dispatch_date']).dt.days
shipments['delivery_days_corrected_flag'] = shipments['delivery_days_original'].ne(shipments['delivery_days']) & shipments['delivery_days_original'].notna()
shipments.loc[shipments['actual_delivery_date'].isna(), 'delivery_days'] = pd.NA

In [12]:
transformed = {
    'stg_customers': customers,
    'stg_customer_reviews': reviews,
    'stg_marketing_campaigns': campaigns,
    'stg_orders': orders,
    'stg_order_items': order_items,
    'stg_payments': payments,
    'stg_products': products,
    'stg_returns': returns,
    'stg_shipments': shipments,
}

## Menyimpan hasil transformasi

In [13]:
for table_name, df in transformed.items():
    output_file = output_dir / f'{table_name}.parquet'
    df.to_parquet(output_file, index=False)
    print(f'{table_name}: {df.shape} -> {output_file}')

stg_customers: (25000, 19) -> c:\Projects\dwh_ecommerce\data\transformed\stg_customers.parquet
stg_customer_reviews: (77530, 10) -> c:\Projects\dwh_ecommerce\data\transformed\stg_customer_reviews.parquet
stg_marketing_campaigns: (151, 15) -> c:\Projects\dwh_ecommerce\data\transformed\stg_marketing_campaigns.parquet
stg_orders: (100000, 19) -> c:\Projects\dwh_ecommerce\data\transformed\stg_orders.parquet
stg_order_items: (254331, 10) -> c:\Projects\dwh_ecommerce\data\transformed\stg_order_items.parquet
stg_payments: (100000, 9) -> c:\Projects\dwh_ecommerce\data\transformed\stg_payments.parquet
stg_products: (550, 16) -> c:\Projects\dwh_ecommerce\data\transformed\stg_products.parquet
stg_returns: (12075, 11) -> c:\Projects\dwh_ecommerce\data\transformed\stg_returns.parquet
stg_shipments: (89681, 13) -> c:\Projects\dwh_ecommerce\data\transformed\stg_shipments.parquet


## Validasi hasil transformasi

In [14]:
validation_summary = pd.DataFrame([
    {
        'table': table_name,
        'rows': len(df),
        'columns': len(df.columns),
        'missing_values': int(df.isna().sum().sum()),
        'duplicate_rows': int(df.duplicated().sum()),
    }
    for table_name, df in transformed.items()
])
display(validation_summary)

,table,rows,columns,missing_values,duplicate_rows
0,stg_customers,25000,19,2300,0
1,stg_customer_reviews,77530,10,0,0
2,stg_marketing_campaigns,151,15,0,0
3,stg_orders,100000,19,0,0
4,stg_order_items,254331,10,0,0
5,stg_payments,100000,9,0,0
6,stg_products,550,16,5,0
7,stg_returns,12075,11,0,0
8,stg_shipments,89681,13,6963,0
